# 디스플레이 공정 제품 이상 여부 판별: 소프트 보팅 최종 모형

- LG Aimers 5기 제품 이상 여부 판별 과제의 최종 제출 모형
- 공정 4단계(Dam, AutoClave, Fill1, Fill2) 자료로 제품의 정상(Normal), 불량(AbNormal) 분류

- 필요한 파일: `train.csv`, `test.csv`, `submission.csv` (대회 제공 자료, 비공개)
- 필요한 패키지: `pycaret[full]`, `catboost`, `xgboost`, `lightgbm`
- 순서: 밀린 열 복구 → 판정 변수, 장비 인코딩 → 값이 하나뿐인 변수 제거 → 변수 6개 선택 → 5개 모형 튜닝(F1 기준)과 소프트 보팅 → 검증 자료로 분류 기준값 선택 → 시험 자료 예측과 공정 간 불일치 행 보정

## 1. 준비

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    make_scorer
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency
from statsmodels.tools.tools import add_constant
from pycaret.classification import *

warnings.filterwarnings('ignore')
RANDOM_STATE = 110

## 2. 자료 불러오기

In [ ]:
# 대회 제공 자료
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
test_df = test_df.drop(['Set ID', 'target'], axis=1)  # 식별 번호와 빈 결과변수 열 제외

## 3. 전처리

### 3.1. 밀린 열 복구

- 일부 행은 특정 변수부터 값이 한 칸씩 밀려 기록되어 있어, 밀린 값을 제자리로 옮김
- X 좌표의 "OK"는 결측으로 처리

In [ ]:
# X 좌표의 "OK"를 결측으로 변환

columns_to_replace_ok = [
    'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
    'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
    'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'
]

# 학습 자료
for column in columns_to_replace_ok:
    train_df[column] = train_df[column].replace('OK', np.nan).astype(float)

# 시험 자료
for column in columns_to_replace_ok:
    test_df[column] = test_df[column].replace('OK', np.nan).astype(float)

In [ ]:
def shift_and_fill(data, start_col, end_col, step):
    # start_col이 결측인 행: start_col~end_col 구간 값을 step 칸씩 앞으로 당기고 마지막 칸은 결측

    start_idx = data.columns.get_loc(start_col)

    end_idx = data.columns.get_loc(end_col)

    missing_indices = data[data[start_col].isnull()].index

    for idx in missing_indices:
        current_idx = start_idx

        while current_idx <= end_idx - step:
            next_idx = current_idx + step
            data.iloc[idx, current_idx] = data.iloc[idx, next_idx]
            current_idx = next_idx

        data.iloc[idx, end_idx] = np.nan

In [ ]:
# Dam
shift_and_fill(train_df, 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
               'WorkMode Collect Result_Dam', 3)
shift_and_fill(test_df, 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
               'WorkMode Collect Result_Dam', 3)
# Fill1
shift_and_fill(train_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1",
               'WorkMode Collect Result_Fill1', 3)
shift_and_fill(test_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1",
               'WorkMode Collect Result_Fill1', 3)

# Fill2
shift_and_fill(train_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2",
               'WorkMode Collect Result_Fill2', 3)
shift_and_fill(test_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2",
               'WorkMode Collect Result_Fill2', 3)

In [ ]:
# 밀린 뒤 비게 된 마지막 열(WorkMode) 채우기

# 학습 자료
train_df['WorkMode Collect Result_Fill1'] = 7
train_df['WorkMode Collect Result_Dam'] = 7
train_df['WorkMode Collect Result_Fill2'] = 0

# 시험 자료
test_df['WorkMode Collect Result_Fill1'] = 7
test_df['WorkMode Collect Result_Dam'] = 7
test_df['WorkMode Collect Result_Fill2'] = 0

### 3.2. 판정 변수 인코딩

- 판정 결과 변수: OK → 1, NG와 결측 → 0

In [ ]:
def process_ok_columns(df, col):
    df[col].replace('OK', 1, inplace=True)

    df[col].fillna(0, inplace=True)

    df[col].replace('NG', 0, inplace=True)

cols = ['Chamber Temp. Judge Value_AutoClave', 'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave']
process_ok_columns(train_df, cols[0])
process_ok_columns(test_df,  cols[0])
process_ok_columns(train_df, cols[1])
process_ok_columns(test_df,  cols[1])

### 3.3. 장비 인코딩과 공정 간 불일치 확인

- 장비 이름: #1 → 0, #2 → 1
- 여러 공정에 반복 기록된 식별 변수(Model.Suffix, Workorder, Receip No, PalletID, Production Qty, Equipment)의 공정 간 불일치 행 확인

In [ ]:
# 1) 장비 인코딩: #1 → 0, #2 → 1
def convert_equipment_values(row):
    if '#1' in str(row):
        return 0
    elif '#2' in str(row):
        return 1
    return row  # 둘 다 아니면 원래 값

train_df['Equipment_Dam'] = train_df['Equipment_Dam'].apply(convert_equipment_values)
train_df['Equipment_Fill1'] = train_df['Equipment_Fill1'].apply(convert_equipment_values)
train_df['Equipment_Fill2'] = train_df['Equipment_Fill2'].apply(convert_equipment_values)

test_df['Equipment_Dam'] = test_df['Equipment_Dam'].apply(convert_equipment_values)
test_df['Equipment_Fill1'] = test_df['Equipment_Fill1'].apply(convert_equipment_values)
test_df['Equipment_Fill2'] = test_df['Equipment_Fill2'].apply(convert_equipment_values)

# 공정마다 반복 기록된 변수 묶음
equipment_cols = ['Equipment_Dam', 'Equipment_Fill1', 'Equipment_Fill2']
model_suffix_cols = [col for col in train_df.columns if "Model.Suffix" in col]
workorder_cols = [col for col in train_df.columns if "Workorder" in col]
receip_no_cols = [col for col in train_df.columns if "Receip No Collect Result" in col]
pallet_id_cols = [col for col in train_df.columns if "PalletID" in col]
production_qty_cols = [col for col in train_df.columns if "Production" in col]

columns_groups = {
    'Model Suffix': model_suffix_cols,
    'Workorder': workorder_cols,
    'Receip No': receip_no_cols,
    'Pallet ID': pallet_id_cols,
    'Production Qty': production_qty_cols,
    'Equipment': equipment_cols
}


# 2) 공정 간 값이 다른 행 찾기
def find_differences(data, columns):
    if not columns:
        return None
    comparison_df = data[columns]
    diff_rows = comparison_df[comparison_df.nunique(axis=1) > 1]  # 행 안에서 값이 2개 이상이면 불일치
    print(f'✅ <불일치한 경우> 총 {len(diff_rows)}개 rows가 불일치함. 비교한 {len(diff_rows.columns)}개 columns 정보: {diff_rows.columns.to_list()}')
    return

def remove_inconsistent_rows(data, columns_groups):
    for key, cols in columns_groups.items():
        diff_rows = find_differences(data, cols)
        if diff_rows is not None:
            data.drop(diff_rows.index, inplace=True)
    return data

train_df = remove_inconsistent_rows(train_df, columns_groups)


# 3) 중복 열 제거: 이후 변수 선택 단계에서 필요한 열만 남기므로 실행하지 않음
'''
remove_cols = ['Model.Suffix_AutoClave', 'Model.Suffix_Fill1', 'Model.Suffix_Fill2',
               'Equipment_Fill1', 'Equipment_Fill2',
               'Workorder_AutoClave', 'Workorder_Fill1', 'Workorder_Fill2',
               'Receip No Collect Result_Fill1', 'Receip No Collect Result_Fill2',
               'Production Qty Collect Result_Fill1', 'Production Qty Collect Result_Fill2',
               'PalletID Collect Result_Fill1','PalletID Collect Result_Fill2']

print("삭제 전:")
print(train_df.shape)
print(test_df.shape)

train_df = train_df.drop(columns=remove_cols)
test_df  = test_df.drop(columns=remove_cols)

print("삭제 후:")
print(train_df.shape)
print(test_df.shape)
'''

### 3.4. 값이 하나뿐인 변수 제거

- 모든 행에 같은 값만 있는 변수는 분류에 정보가 없어 제거
- 장비 #1, #2 집단 모두에서 값이 하나뿐인 변수도 제거

In [ ]:
print("삭제 전:")
print(train_df.shape)
print(test_df.shape)


# 1) 값이 하나 이하인 변수
unique_counts = train_df.nunique()
tr_cols_to_drop = unique_counts[unique_counts <= 1].index

unique_counts = test_df.nunique()
te_cols_to_drop = unique_counts[unique_counts <= 1].index

train_df = train_df.drop(columns=tr_cols_to_drop)
test_df  = test_df.drop(columns=te_cols_to_drop)



# 2) 장비 #1, #2 집단 모두에서 값이 하나뿐인 변수
def remove_single_value_columns(df):
    group_0 = df[df['Equipment_Dam'] == 0]
    group_1 = df[df['Equipment_Dam'] == 1]

    def find_single_value_columns(data):
        columns_to_remove = []
        for col in data.columns:
            if col != 'Equipment_Dam' and len(data[col].unique()) == 1:
                columns_to_remove.append(col)
        return columns_to_remove

    columns_to_remove_0 = find_single_value_columns(group_0)
    columns_to_remove_1 = find_single_value_columns(group_1)

    common_columns_to_remove = list(set(columns_to_remove_0) & set(columns_to_remove_1))

    return df.drop(columns=common_columns_to_remove, inplace=False), common_columns_to_remove


train_df, train_removed_cols = remove_single_value_columns(train_df)
test_df = test_df.drop(columns=train_removed_cols, errors='ignore')


print("삭제 후:")
print(train_df.shape)
print(test_df.shape)

### 3.5. 남은 "OK" 처리와 전체 결측 변수 제거

In [ ]:
print("삭제 전:")
print(train_df.shape)
print(test_df.shape)


# 1) 'OK'를 결측으로 변환
train_df.replace('OK', np.nan, inplace=True)
test_df.replace('OK', np.nan, inplace=True)


# 2) 모든 값이 결측인 변수 제거
def remove_all_missing_columns(data):
    is_all_missing = data.isnull().all()
    cleaned_data = data.loc[:, ~is_all_missing]
    return cleaned_data

train_df = remove_all_missing_columns(train_df)
test_df = remove_all_missing_columns(test_df)


print("삭제 후:")
print(train_df.shape)
print(test_df.shape)

## 4. 변수 선택

- 시도 과정의 변수 중요도 결과로 고른 6개 변수만 사용

In [ ]:
train_df = train_df[['Production Qty Collect Result_Dam','Workorder_AutoClave', 'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill1',
                     'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill1',
                     'DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1','DISCHARGED SPEED OF RESIN Collect Result_Fill1'
                     ,'target']]
test_df = test_df[['Production Qty Collect Result_Dam','Workorder_AutoClave', 'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill1',
                     'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill1',
                     'DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1','DISCHARGED SPEED OF RESIN Collect Result_Fill1'
                     ]]

## 5. 모형

- 학습 80%, 검증 20% 층화 분할
- PyCaret으로 GBC, CatBoost, 랜덤 포레스트, LightGBM, XGBoost를 각각 F1 기준 튜닝(10겹 교차검증, 무작위 탐색 10회)
- 5개 모형의 예측확률을 평균하는 소프트 보팅

In [ ]:
import optuna
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import f1_score, make_scorer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np
from collections import defaultdict
from joblib import Parallel, delayed

In [ ]:
train_split_df, valid_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["target"],
    random_state=RANDOM_STATE,
)

y_train = train_split_df['target']
X_train = train_split_df.drop('target', axis=1)

y_valid = valid_df['target']
X_valid = valid_df.drop('target', axis=1)



clf = setup(data=X_train, target=y_train, session_id=RANDOM_STATE, n_jobs=4)

kfold = 10

def my_tune_model(model_name, fold, n_iter, optimize):
  m = create_model(model_name, fold = kfold)
  return tune_model(m, fold = kfold, n_iter = n_iter, optimize = optimize)


model_list = []
for model_name in ['gbc','catboost', 'rf', 'lightgbm', 'xgboost']:
  model_list.append(my_tune_model(model_name, fold = kfold, n_iter =10, optimize = 'F1'))

kfold = 10
blended_model = blend_models(estimator_list = model_list,
                       fold = kfold,
                       method = 'soft',
                       optimize='F1'
                       )

## 6. 검증 자료로 분류 기준값 선택

- 불량 비율이 낮아 기준값 0.5 대신 검증 자료 F1(불량 기준)이 가장 높은 기준값 선택

In [ ]:
# 검증 자료 예측
y_pred = predict_model(blended_model, data = X_valid)
thresholds = np.arange(0.50, 1.00, 0.001)

f1_scores = []

for threshold in thresholds:
    y_valid_pred = ['Normal' if score >= threshold else 'AbNormal' for score in y_pred['prediction_score']]

    f1 = f1_score(y_valid, y_valid_pred, pos_label='AbNormal')
    f1_scores.append((threshold, f1))

In [ ]:
# F1이 가장 높은 기준값
best_threshold, best_f1_score = max(f1_scores, key=lambda x: x[1])

print(f"Best Threshold: {best_threshold:.3f}, Best F1 Score: {best_f1_score:.4f}")

## 7. 시험 자료 예측

- 검증 자료에서 고른 기준값(0.836)으로 분류

In [ ]:
pred = predict_model(blended_model, data = test_df)
pred['y_test_pred'] = ['Normal' if score >= 0.836 else 'AbNormal' for score in pred['prediction_score']]

y_test_pred = ['Normal' if score >= 0.836 else 'AbNormal' for score in pred['prediction_score']]

# 제출 양식
df_sub = pd.read_csv("submission.csv")

df_sub["target"] = pred['y_test_pred']

df_sub.to_csv("submission.csv", index=False)

In [ ]:
pred

### 7.1. 공정 간 불일치 행 보정

- 여러 공정에서 같아야 하는 식별 변수 값이 서로 다른 행은 불량으로 분류
- 제출 당시에는 아래처럼 해당 행 번호를 직접 지정

In [ ]:
# 공정 간 불일치 행 번호
Abnormal_rows = [64, 562, 1460, 1530, 1892, 2505, 2710, 3457, 3682, 3732, 4928, 4932, 6092,
 7001, 7287, 7666, 7836, 8253, 8898, 10989, 12439, 12585, 12844, 14756, 15180, 15406, 15811, 15964]


for idx in Abnormal_rows:
    if pred['y_test_pred'][idx] != "AbNormal":
        pred['y_test_pred'][idx] = "AbNormal"
    else:
        print(f'{idx} is passed')

In [ ]:
len(Abnormal_rows)

In [ ]:
sub_data = pd.read_csv('submission.csv')

# 보정한 예측으로 제출 파일 저장
sub_data['target'] = pred['y_test_pred']
sub_data.to_csv('submission.csv', index=False)

## 8. 예측 범주 분포

In [ ]:
def analyze_class_distribution(y_pred):
    unique_classes, counts = np.unique(y_pred, return_counts=True)

    total_count = len(y_pred)
    class_counts = dict(zip(unique_classes, counts))
    class_ratios = {cls: count / total_count for cls, count in class_counts.items()}

    results = {
        'class_counts': class_counts,
        'class_ratios': class_ratios
    }

    return results

In [ ]:
analysis = analyze_class_distribution(pred['y_test_pred'])

print("클래스별 개수:", analysis['class_counts'])
print("클래스별 비율:", analysis['class_ratios'])

## 참고: 불일치 행 자동 탐지

- 7.1에서 직접 지정한 행 번호를 코드로 찾는 방법
- `test_pre.csv`: 3.1 전처리만 수행한 시험 자료

In [ ]:
new_test_df = pd.read_csv("test_pre.csv")

# 공정마다 반복 기록된 변수 묶음
equipment_cols = ['Equipment_Dam', 'Equipment_Fill1', 'Equipment_Fill2']
model_suffix_cols = [col for col in new_test_df.columns if "Model.Suffix" in col]
workorder_cols = [col for col in new_test_df.columns if "Workorder" in col]
receip_no_cols = [col for col in new_test_df.columns if "Receip No Collect Result" in col]
pallet_id_cols = [col for col in new_test_df.columns if "PalletID Collect Result" in col]
production_qty_cols = [col for col in new_test_df.columns if "Production Qty Collect Result" in col]

columns_groups = {
    'Equipment': equipment_cols,
    'Model Suffix': model_suffix_cols,
    'Workorder': workorder_cols,
    'Receip No': receip_no_cols,
    'Pallet ID': pallet_id_cols,
    'Production Qty': production_qty_cols
}

# 장비 인코딩: #1 → 0, #2 → 1
def convert_equipment_values(row):
    if '#1' in str(row):
        return 0
    elif '#2' in str(row):
        return 1
    return row  # 둘 다 아니면 원래 값

new_test_df['Equipment_Dam'] = new_test_df['Equipment_Dam'].apply(convert_equipment_values)
new_test_df['Equipment_Fill1'] = new_test_df['Equipment_Fill1'].apply(convert_equipment_values)
new_test_df['Equipment_Fill2'] = new_test_df['Equipment_Fill2'].apply(convert_equipment_values)


# 불일치 행 번호 (중복 제거)
unique_indices = set()

# 공정 간 값이 다른 행 찾기
def find_differences(data, columns):
    if not columns:
        return None
    comparison_df = data[columns]
    diff_rows = comparison_df[comparison_df.nunique(axis=1) > 1]  # 행 안에서 값이 2개 이상이면 불일치
    print(f'✅ <불일치한 경우> 총 {len(diff_rows)}개 rows가 불일치함. 비교한 {len(diff_rows.columns)}개 columns 정보: {diff_rows.columns.to_list()}')
    return diff_rows

# 변수 묶음별 불일치 행 번호 수집
def check_and_collect_indices(data, columns_groups):
    for key, cols in columns_groups.items():
        if cols:
            diff_rows = find_differences(data, cols)
            if diff_rows is not None:
                unique_indices.update(diff_rows.index)

check_and_collect_indices(new_test_df, columns_groups)
unique_indices_list = sorted(list(unique_indices))


# 모형 예측 결과
df_sub = pd.DataFrame(columns=['target'])
df_sub['target'] = y_test_pred

# 불일치 행은 불량으로
for idx in unique_indices_list:
  if idx in df_sub.index:
    df_sub.loc[idx, 'target'] = 'AbNormal'

y_test_pred = df_sub['target'].tolist()

In [ ]:
columns_groups

In [ ]:
unique_indices_list